[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_ImageClassification.ipynb)

# Benchmark: Image Classification

Scores a pretrained image classifier against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="imageclassification")`. The report's
`accuracy` is the model's top-1 label, the same as every other task, with `top5Accuracy`
reported alongside it (pass `top_k=` to resize that window).

**Dataset**: a small [ImageNet](https://www.image-net.org/)-labeled sample bundled by John Snow
Labs for its own example notebooks (each filename names its ground-truth ImageNet-1k class, e.g.
`hippopotamus.JPEG`) -- the natural pairing for a model trained on ImageNet-1k.

**Model**: `ViTForImageClassification.pretrained()` (default:
`image_classifier_vit_base_patch16_224`), trained on ImageNet-1k.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import ImageAssembler
from sparknlp.annotator import ViTForImageClassification
from pyspark.ml import Pipeline
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
import shutil
import urllib.request
import os

urllib.request.urlretrieve(
    "https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/resources/en/images/images.zip",
    "/tmp/images.zip")
shutil.unpack_archive("/tmp/images.zip", "/tmp/images", "zip")

# Ground truth: JSL derived each filename from the image's actual ImageNet-1k label. Note that
# ImageNet-1k class names are occasionally compound/comma-separated synonyms (e.g. "hippopotamus,
# hippo, river horse, Hippopotamus amphibius") -- the gold string below has to match the model's
# output exactly, synonyms and all.
gold_labels = {
    "egyptian_cat.jpeg": "Egyptian cat",
    "ox.JPEG": "ox",
    "hippopotamus.JPEG": "hippopotamus, hippo, river horse, Hippopotamus amphibius",
    "hen.JPEG": "hen",
    "ostrich.JPEG": "ostrich, Struthio camelus",
    "junco.JPEG": "junco, snowbird",
    "palace.JPEG": "palace",
    "chihuahua.jpg": "Chihuahua",
    "tractor.JPEG": "tractor",
    "bluetick.jpg": "bluetick",
}
image_dir = "/tmp/images/images"
print(len(gold_labels), "images")

10 images

> **Note: attach labels via `withColumn` on the original image read, not
> `spark.createDataFrame` on collected rows.** `ImageAssembler` requires Spark's special
> ImageSchema metadata, which only `spark.read.format("image")` produces -- rebuilding the
> DataFrame from Python-side rows silently drops that metadata and `ImageAssembler` rejects the
> result. Attaching the gold label as a new column on the original read keeps the metadata
> intact.

In [10]:
import pyspark.sql.functions as F

gold_label_map_udf = F.udf(lambda origin: gold_labels[os.path.basename(origin)], "string")
gold_data = spark.read.format("image").option("dropInvalid", True).load(path=image_dir) \
    .withColumn("label", gold_label_map_udf(F.col("image.origin")))

## 2. Build the pipeline

In [12]:
image_assembler = ImageAssembler().setInputCol("image").setOutputCol("image_assembler")
image_classifier = ViTForImageClassification.pretrained() \
    .setInputCols(["image_assembler"]).setOutputCol("class")

pipeline = Pipeline(stages=[image_assembler, image_classifier])
pipeline_model = pipeline.fit(gold_data)

pipeline_model.transform(gold_data).selectExpr("image.origin", "class.result").show(20, truncate=80)

image_classifier_vit_base_patch16_224 download started this may take some time.
Approximate size to download 309 MB

[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[ / ]
[ — ]
[OK!]
+-------------------------------------------+----------------------------------------------------------+
|                                     origin|                                                    result|
+-------------------------------------------+----------------------------------------------------------+
|      file:///tmp/images/images/palace.JPEG|                                                  [palace]|
|file:///tmp/images/images/egyptian_cat.jpeg|                                            [Egyptian cat]|
|file:///tmp/images/images/hippopotamus.JPEG|[hippopotamus, hippo, river horse, Hippopotamus amphibius]|
|         file:///tmp/images/images/hen.JPEG|                                                     [hen]|
|     file:///tmp/images/images/ostrich.JPEG|                               

## 3. Run the benchmark

In [14]:
report = Benchmark.evaluate(
    pipeline_model, gold_data, task="imageclassification", text_col="image", label_col="label")
print(report)

imageclassification accuracy (n=10): accuracy=1.0000, weightedF1=1.0000, weightedPrecision=1.0000, weightedRecall=1.0000
  Chihuahua: f1=1.0000, precision=1.0000, recall=1.0000
  Egyptian cat: f1=1.0000, precision=1.0000, recall=1.0000
  bluetick: f1=1.0000, precision=1.0000, recall=1.0000
  hen: f1=1.0000, precision=1.0000, recall=1.0000
  hippopotamus, hippo, river horse, Hippopotamus amphibius: f1=1.0000, precision=1.0000, recall=1.0000
  junco, snowbird: f1=1.0000, precision=1.0000, recall=1.0000
  ostrich, Struthio camelus: f1=1.0000, precision=1.0000, recall=1.0000
  ox: f1=1.0000, precision=1.0000, recall=1.0000
  palace: f1=1.0000, precision=1.0000, recall=1.0000
  tractor: f1=1.0000, precision=1.0000, recall=1.0000